# Scania APS - Fault Detection Model
**Arkon Manufacturing AI | Module: ML Classification | Department: Truck Fleet**

**Goal:** Train a binary classifier to detect APS failures before they cause truck breakdowns.

**Cost metric (Scania competition standard):**
- Total cost = 10 × FP + 500 × FN
- Optimise classification threshold to minimise total cost, not just accuracy

Approach:
1. Baseline: Logistic Regression
2. Main model: XGBoost Classifier
3. Threshold optimisation
4. MLflow tracking for all runs
5. Save best model

In [ ]:
import sys
from pathlib import Path

# Add notebooks/utils to path (works on Mac, Windows, Linux)
_nb_root = Path('..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)

print('arkon_utils loaded ✓')

## 1. Imports & Configuration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, preds, title in zip(
    axes,
    [preds_xgb, preds_opt],
    ['XGBoost (threshold=0.5)', f'XGBoost (threshold={optimal_threshold:.2f})'],
):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(title)
plt.tight_layout()
save_figure(fig, 'scania_model_confusion_matrices', subfolder=ASSETS)
plt.show()


## 2. Connect to MLflow

In [ ]:
mlflow.set_tracking_uri(get_mlflow_uri())
mlflow.set_experiment('arkon-ml-scania')
print(f'MLflow URI   : {mlflow.get_tracking_uri()}')


## 3. Load Processed Data

In [ ]:
# Load balanced training set and processed test set from notebook 02
df_train = pd.read_csv(PROC_DIR / 'train_scania_processed.csv')
df_test  = pd.read_csv(PROC_DIR / 'test_scania_processed.csv')
feature_names = joblib.load(PROC_DIR / 'feature_names_scania.pkl')

X_train = df_train[feature_names].values
y_train = df_train['label'].values
X_test  = df_test[feature_names].values
y_test  = df_test['label'].values

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Train class balance: {pd.Series(y_train).value_counts().to_dict()}')

## 4. Helper Functions

In [ ]:
# Calculate total business cost using Scania competition formula
def scania_cost(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return COST_FP * fp + COST_FN * fn

# Print full evaluation report including cost metric
def evaluate(model, X, y, threshold=0.5, label='test'):
    proba = model.predict_proba(X)[:, 1]
    preds = (proba >= threshold).astype(int)
    auc   = roc_auc_score(y, proba)
    cost  = scania_cost(y, preds)
    print(f'\n--- {label} (threshold={threshold}) ---')
    print(classification_report(y, preds, target_names=['neg', 'pos']))
    print(f'ROC-AUC:      {auc:.4f}')
    print(f'Scania cost:  {cost:,}')
    return proba, preds, {'auc': auc, 'cost': cost}

## 5. Baseline - Logistic Regression

In [ ]:
ckpt = CheckpointManager(CKPT_DIR)

if ckpt.exists('scania_lr_baseline'):
    lr, meta_lr = ckpt.load_sklearn('scania_lr_baseline')
    print(f'Loaded LR checkpoint | train time: {meta_lr.get("train_time", "?")}')  
else:
    with mlflow.start_run(run_name='baseline_logistic_regression'):
        lr = LogisticRegression(max_iter=500, random_state=RANDOM_STATE, n_jobs=-1)
        with Timer('Logistic Regression training') as t_lr:
            lr.fit(X_train, y_train)

        proba_lr = lr.predict_proba(X_test)[:, 1]
        preds_lr = (proba_lr >= 0.5).astype(int)
        cost_lr  = scania_cost(y_test, preds_lr)
        auc_lr   = roc_auc_score(y_test, proba_lr)

        mlflow.log_params({'model': 'LogisticRegression', 'max_iter': 500})
        mlflow.log_metrics({'cost': cost_lr, 'auc': auc_lr})
        mlflow.log_param('train_time_s', t_lr.seconds)

        meta_lr = {'train_time': t_lr.report(), 'cost': cost_lr, 'auc': round(auc_lr, 4)}
        ckpt.save_sklearn(lr, 'scania_lr_baseline', metadata=meta_lr)
        print(f'LR: cost={cost_lr}  AUC={auc_lr:.4f}  time={t_lr.report()}')


## 6. Main Model - XGBoost

In [ ]:
xgb_params = {
    'n_estimators':     500,
    'max_depth':        6,
    'learning_rate':    0.05,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'eval_metric':      'logloss',
    'random_state':     RANDOM_STATE,
    'n_jobs':           -1,
    'tree_method':      'hist',
    # 'device': 'cuda',  # uncomment to use GPU for XGBoost
}

if ckpt.exists('scania_xgb_v1'):
    xgb, meta_xgb = ckpt.load_sklearn('scania_xgb_v1')
    proba_xgb = xgb.predict_proba(X_test)[:, 1]
    preds_xgb = (proba_xgb >= 0.5).astype(int)
    print(f'Loaded XGB checkpoint | train time: {meta_xgb.get("train_time", "?")}')  
else:
    with mlflow.start_run(run_name='xgboost_v1'):
        xgb = XGBClassifier(**xgb_params)
        with Timer('XGBoost training') as t_xgb:
            xgb.fit(X_train, y_train)

        proba_xgb = xgb.predict_proba(X_test)[:, 1]
        preds_xgb = (proba_xgb >= 0.5).astype(int)
        cost_xgb  = scania_cost(y_test, preds_xgb)
        auc_xgb   = roc_auc_score(y_test, proba_xgb)

        mlflow.log_params(xgb_params)
        mlflow.log_metrics({'cost': cost_xgb, 'auc': auc_xgb})
        mlflow.log_param('train_time_s', t_xgb.seconds)
        mlflow.sklearn.log_model(xgb, 'model')

        meta_xgb = {'train_time': t_xgb.report(), 'cost': cost_xgb, 'auc': round(auc_xgb, 4)}
        ckpt.save_sklearn(xgb, 'scania_xgb_v1', metadata=meta_xgb)
        print(f'XGB: cost={cost_xgb}  AUC={auc_xgb:.4f}  time={t_xgb.report()}')


## 7. Threshold Optimisation

In [ ]:
# Find the threshold that minimises total business cost (10×FP + 500×FN)
thresholds = np.arange(0.1, 0.9, 0.01)
costs = []

for t in thresholds:
    preds_t = (proba_xgb >= t).astype(int)
    costs.append(scania_cost(y_test, preds_t))

# Find optimal threshold with minimum cost
optimal_idx = np.argmin(costs)
optimal_threshold = thresholds[optimal_idx]
optimal_cost = costs[optimal_idx]

print(f'Optimal threshold: {optimal_threshold:.2f}')
print(f'Minimum cost:      {optimal_cost:,}')

# Plot cost vs threshold curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, costs, color='steelblue')
ax.axvline(optimal_threshold, color='red', linestyle='--',
           label=f'Optimal threshold = {optimal_threshold:.2f} (cost = {optimal_cost:,})')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Total Business Cost')
ax.set_title('Threshold Optimisation - Scania Cost Metric')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Log optimised threshold run to MLflow as a separate experiment
with mlflow.start_run(run_name='xgboost_optimised_threshold'):

    _, preds_opt, metrics_opt = evaluate(
        xgb, X_test, y_test,
        threshold=optimal_threshold, label='XGB optimised'
    )

    # Log all parameters including the optimised threshold
    mlflow.log_params(xgb_params)
    mlflow.log_param('threshold', optimal_threshold)
    mlflow.log_param('smote', True)
    mlflow.log_metrics(metrics_opt)
    mlflow.xgboost.log_model(xgb, 'model')

## 8. Confusion Matrix

In [ ]:
# Plot confusion matrices for default and optimised thresholds side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, title in zip(
    axes,
    [preds_xgb, preds_opt],
    ['XGBoost (threshold=0.5)', f'XGBoost (threshold={optimal_threshold:.2f})']
):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['neg', 'pos'],
                yticklabels=['neg', 'pos'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)

plt.suptitle('Confusion Matrices - Scania APS Test Set', fontsize=13)
plt.tight_layout()
plt.show()

## 9. ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for proba, label in [(proba_lr, 'Logistic Regression'), (proba_xgb, 'XGBoost')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curve - Scania APS Failure')
ax.legend()
save_figure(fig, 'scania_model_roc_curve', subfolder=ASSETS)
plt.show()


## 10. Feature Importance

In [ ]:
importance = pd.Series(xgb.feature_importances_, index=feature_names)
top20 = importance.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
top20.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('XGBoost - Top 20 Feature Importances (Scania APS)')
plt.tight_layout()
save_figure(fig, 'scania_model_feature_importance', subfolder=ASSETS)
plt.show()


## 11. Save Best Model

In [ ]:
# Save final model and optimal threshold for use in Streamlit app
joblib.dump(xgb, MODELS_DIR / 'scania_xgb.pkl')
joblib.dump(optimal_threshold, MODELS_DIR / 'scania_threshold.pkl')

# Save test predictions for Tableau dashboard
pd.DataFrame({
    'true_label': y_test,
    'proba_failure': proba_xgb,
    'pred_label_05': preds_xgb,
    'pred_label_opt': preds_opt
}).to_csv(PROC_DIR / 'test_scania_predictions.csv', index=False)

print(f'Saved: scania_xgb.pkl, scania_threshold.pkl')
print(f'Optimal threshold: {optimal_threshold:.2f}')

## 12. Results Summary

| Model | AUC | Cost (0.5) | Cost (optimal) |
|-------|-----|------------|----------------|
| Logistic Regression | - | - | - |
| XGBoost v1 | - | - | - |

*(Fill in after running)*

**Key insight:** Threshold optimisation reduces business cost significantly
by trading some false positives (cheap) for fewer false negatives (expensive).

View all runs: **http://127.0.0.1:5001** → experiment: arkon-ml-scania

**Next step:** Integrate into Streamlit page `03_ml_scania.py`.

In [ ]:
# ── Model comparison with training times ─────────────────────────────
import pandas as pd
results = pd.DataFrame([
    {'Model': 'Logistic Regression', **meta_lr},
    {'Model': 'XGBoost v1',          **meta_xgb},
])
results = results[['Model', 'cost', 'auc', 'train_time']]
results.columns = ['Model', 'Scania Cost', 'AUC', 'Train Time']
print(results.to_string(index=False))
